# VecDB HNSW Index & efsearch Tuning
Demonstrate how to create an Oracle VecDB HNSW index and control recall/latency per query by tuning `efsearch`, the number of candidates evaluated during search.


## 1. Scenario Overview
This quickstart walks through:
1. Authenticating with VecDB using `.env` credentials and creating a tiny demo table.
2. Building an explicit HNSW index (in-memory graph) with custom neighbor and `efConstruction` settings.
3. Running multiple searches that highlight how `efsearch` = `number_of_candidates` impacts recall vs. latency.


## 2. Setup
Install or upgrade the VecDB SDK stack (`%pip` cell below), then hydrate configuration only once so every other cell reuses the same `OracleVecDB` client. The second code cell also enables optional SSL overrides for self-signed test clusters and prints whichever host/user values were pulled from `.env`.

In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas sentence-transformers

In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading VecDB environment variables...')
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
VECDB_USER = os.getenv('VECDB_USERNAME')
VECDB_PASSWORD = os.getenv('VECDB_PASSWORD')
vecdb_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print('Resolved REST endpoint:', resolved_host)
print('Resolved user:', VECDB_USER)


config_kwargs = {"rest_url": resolved_host}
if vecdb_access_token:
    vecdb_config_kwargs["access_token"] = vecdb_access_token
else:
    config_kwargs["username"] = VECDB_USER
    config_kwargs["password"] = VECDB_PASSWORD
config = Configuration(**config_kwargs)

if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification for self-signed certificates.')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Auth method:', auth_method)
print('VecDB client ready for HNSW walkthrough.')


## 3. Create Demo Table
Provision a lightweight dense-vector table, describe the embeddings helper that the rest of the notebook calls, and seed a handful of descriptive records so you can experiment safely. The following code cell (plus the `Embedding Strategy` note) handles:
1. Creating or resetting the demo table with manual indexing.
2. Resolving whether to download a SentenceTransformer model or fall back to deterministic vectors.
3. Defining `embed_text()` and populating metadata-rich rows that act as our mini dataset.

### Embedding Strategy
- By default the notebook keeps dependencies light by generating deterministic demo vectors.
- Set `USE_HOSTED_EMBEDDINGS=true` (and optionally `EMBED_MODEL_NAME=all-MiniLM-L12-v2`) to download a SentenceTransformer model and embed real text.
- `embed_text()` returns the dense vector used both for table rows and for the query examples, so you can swap in any embedding provider without changing the indexing logic.

In [ ]:
from uuid import uuid4
from random import Random


def env_or_default(name, default):
    value = os.getenv(name)
    return value.strip() if value and value.strip() else default


def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


TABLE = env_or_default('HNSW_TABLE', 'HNSW_EFSEARCH_DEMO')
print('Target table:', TABLE)

vecdb.create_vector_table(
    name=TABLE,
    comment='HNSW / efsearch tutorial table',
    annotations={'TITLE': 'string', 'CATEGORY': 'string', 'SUMMARY': 'string'},
)
print('Created vector table. HNSW index is created explicitly in the next section.')

USE_HOSTED_EMBEDDINGS = env_or_default('USE_HOSTED_EMBEDDINGS', 'false').lower() == 'true'
MODEL_NAME = env_or_default('EMBED_MODEL_NAME', 'all-MiniLM-L12-v2')
MODEL_NAME = 'all-MiniLM-L12-v2'
DEFAULT_DIM = int(env_or_default('DEMO_VECTOR_DIM', '16'))

if USE_HOSTED_EMBEDDINGS:
    print('Loading SentenceTransformer model:', MODEL_NAME)
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer(MODEL_NAME)

    def embed_text(text: str) -> list[float]:
        return embedder.encode(text, normalize_embeddings=True).tolist()
else:
    print('Using deterministic demo embeddings (no model download).')

    def embed_text(text: str, dim: int = DEFAULT_DIM) -> list[float]:
        rng = Random(abs(hash(text)) % (2**32))
        return [rng.random() for _ in range(dim)]

seed_docs = [
    {'title': 'Global marketing brief', 'category': 'marketing', 'text': 'A detailed omni-channel launch plan highlighting positioning guardrails and localization guidance for regional field teams.'},
    {'title': 'Predictive maintenance plan', 'category': 'operations', 'text': 'Maintenance engineers review sensor baselines, anomaly scoring, and rollout sequencing for robotics-enabled warehouses.'},
    {'title': 'Customer onboarding workflow', 'category': 'success', 'text': 'Step-by-step enablement journey describing kickoff workshops, sandbox provisioning, and success metrics for enterprise customers.'},
    {'title': 'Risk mitigation summary', 'category': 'finance', 'text': 'Quarterly memo covering liquidity scenarios, hedging triggers, and board escalation procedures during macro volatility.'},
    {'title': 'Cybersecurity executive briefing', 'category': 'security', 'text': 'Security leaders share zero-trust adoption lessons, identity segmentation tactics, and tabletop exercises for hybrid teams.'},
    {'title': 'AI product ethics review', 'category': 'governance', 'text': 'Packet outlining bias testing strategy, human-in-the-loop guardrails, and customer communication best practices.'},
    {'title': 'Sustainability roadmap', 'category': 'strategy', 'text': 'Roadmap describing energy-efficiency retrofits, supplier scorecards, and long-horizon carbon removal investments.'},
    {'title': 'Sales kickoff agenda', 'category': 'revenue', 'text': 'Full-week agenda with keynote summaries, industry breakouts, competitive battle cards, and certification labs.'},
    {'title': 'Zero-trust playbook', 'category': 'security', 'text': 'Hands-on playbook for rolling out device trust scores, adaptive MFA, and lateral movement detection across hybrid networks.'},
    {'title': 'Incident tabletop recap', 'category': 'security', 'text': 'Summary of a ransomware simulation including executive communications, SOC escalation triggers, and forensic follow-ups.'}
]

rows = [
    {
        'id': str(uuid4()),
        'dense_vector': embed_text(doc['text']),
        'metadata': {
            'TITLE': doc['title'],
            'CATEGORY': doc['category'],
            'SUMMARY': doc['text'],
        },
    }
    for doc in seed_docs
]
vecdb.upsert_vectors(table_name=TABLE, vectors=rows)
status_msg = (
    f"Seeded {len(rows)} rows using hosted embeddings."
    if USE_HOSTED_EMBEDDINGS
    else f"Seeded {len(rows)} rows with deterministic text embeddings."
)
print(status_msg)

## 4. Build an HNSW Index
Capture the exact HNSW configuration you want to showcase and submit it through `vecdb.create_index()`. This cell spells out every knob (organization, metric, neighbors, `efConstruction`) so the output job payload can be reused later in the monitoring cells.

In [ ]:
index_params = {
    'vector_index_params': {
        'organization': 'INMEMORY GRAPH',
        'distance_metric': 'COSINE',
        'advanced_params': {
            'neighbors': 32,
            'efConstruction': 200,
        },
    },
}

print('Submitting HNSW index job...')
job = vecdb.create_index(table_name=TABLE, index_params=index_params)
print('Index job:', job)


### Monitor the asynchronous index job
The next two helper cells show how to poll the VecDB job roster:
- `vecdb.describe_index_job(...)` surfaces the job header (state, creator, timestamps, and links).
- `vecdb.get_index_job_log(...)` fetches the attached jobfile so you can capture warnings/errors for demos or docs.
Update the job IDs if you rerun the notebook so they match the latest submission.

In [ ]:
jobs = vecdb.list_index_jobs()
if jobs.items:
    latest_job = jobs.items[0]

    job_name = getattr(latest_job, 'job_name', None)
    response = vecdb.describe_index_job(index_job_name=job_name)
    print(response)


In [ ]:
jobs = vecdb.list_index_jobs()
if jobs.items:
    latest_job = jobs.items[0]

    job_name = getattr(latest_job, 'job_name', None)
    response = vecdb.get_index_job_log(index_job_name=job_name)
    print(response)

## 5. Query With Different `efsearch` Values
`efsearch` is literally the HNSW **number of candidates** (beam width) that VecDB evaluates while searching. Higher values usually improve recall but consume more time/CPU; smaller values respond faster but may miss neighbors.

Both searches below keep `top_k=3`. The only change is the candidate beam: `efsearch=4` vs `efsearch=256`. Use `advanced_options={'idx_parameters': {'efsearch': value}}` on `vecdb.query()` to override the default per request.

In [ ]:
QUERY_TEXT = env_or_default('HNSW_QUERY_TEXT', 'Security leaders share zero-trust lessons for hybrid teams')
print('Query text:', QUERY_TEXT)
print('Note: Recall differences are easier to see on larger dataset; this demo keeps only ~10 docs for example')
query_vector = embed_text(QUERY_TEXT)

low_recall = vecdb.query(
    table_name=TABLE,
    query_by={'vector': query_vector},
    top_k=3,
    advanced_options={'idx_parameters': {'efsearch': 4}},
)

high_recall = vecdb.query(
    table_name=TABLE,
    query_by={'vector': query_vector},
    top_k=3,
    advanced_options={'idx_parameters': {'efsearch': 256}},
)

print('efsearch=4 results (same top_k=3):')
for item in query_items(low_recall):
    print(
        result_metadata(item).get('TITLE'),
        '\n  category=', result_metadata(item).get('CATEGORY'),
        '\n  distance=', result_distance(item)
    )

print('\nefsearch=256 results (same top_k=3, wider candidate beam):')
for item in query_items(high_recall):
    print(
        result_metadata(item).get('TITLE'),
        '\n  category=', result_metadata(item).get('CATEGORY'),
        '\n  distance=', result_distance(item)
    )


## 6. Cleanup (Optional)
Drop the index and table if you want to re-run the notebook from a clean state. These calls are wrapped in `try/except` so you can safely re-run even if the resources were already removed.

In [ ]:
try:
    vecdb.drop_index(table_name=TABLE, index_params={"index_type": "all"})
    print('Dropped HNSW index on', TABLE)
except Exception as exc:
    print('Index cleanup skipped:', exc)

try:
    vecdb.drop_vector_table(name=TABLE)
    print('Dropped table', TABLE)
except Exception as exc:
    print('Table cleanup skipped:', exc)
